# **Testing emMorph**

In [ ]:
"""IMPORTS AND PATHS"""

from pathlib import Path
import json
import sys
import contextlib
import io
import os
import shutil

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "emmorph_audit":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from helpers.emmorph_pos import emmorph_analyse_many
from helpers.emmorph_test import (
    get_best_stem,
    clean_suffix_df,
    get_suffix_pairs_from_best_matching_analysis,
    classify_preverb_analyses,
)
from helpers.suffix_variants import normalize_suffix, SUFFIX_GLOSSES

JSON_ROOT = PROJECT_ROOT / "json_files_gold_thesis"
STRUCTURED_CSV = PROJECT_ROOT / "input_files" / "etymology_database_gold_thesis.csv"

OUT_DIR = PROJECT_ROOT / "emmorph_audit" / "emmorph_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# HFST / emMorph paths
HU_HFSTOL = Path(os.environ.get(
    "HU_HFSTOL",
    PROJECT_ROOT / "input_files" / "hu.hfstol",
))

hfst_lookup_env = os.environ.get("HFST_LOOKUP")
if hfst_lookup_env:
    HFST_LOOKUP = Path(hfst_lookup_env)
else:
    hfst_from_path = shutil.which("hfst-lookup.exe") or shutil.which("hfst-lookup")
    if hfst_from_path:
        HFST_LOOKUP = Path(hfst_from_path)
    else:
        HFST_LOOKUP = Path.home() / "hfst" / "bin" / "hfst-lookup.exe"


print("Project root:", PROJECT_ROOT)
print("Gold JSON folder:", JSON_ROOT)
print("Structured CSV:", STRUCTURED_CSV)
print("Output folder:", OUT_DIR)
print("HFST lookup:", HFST_LOOKUP, HFST_LOOKUP.exists())
print("hu.hfstol:", HU_HFSTOL, HU_HFSTOL.exists())

for path in [JSON_ROOT, STRUCTURED_CSV, HFST_LOOKUP, HU_HFSTOL]:
    if not path.exists():
        raise FileNotFoundError(path)


In [ ]:

uesz_df = pd.read_csv(STRUCTURED_CSV, encoding="utf-8-sig", dtype=str).fillna("")


rows_with_pos = uesz_df[uesz_df["pos"].astype(str).str.strip() != ""].copy()


pos_by_entry_file = {}

for entry_file, group in rows_with_pos.groupby("entry_file"):
    pos_labels = []

    for pos_value in group["pos"]:
        parts = str(pos_value).split(";")

        for part in parts:
            part = part.strip()
            if part:
                pos_labels.append(part)

    pos_by_entry_file[entry_file] = sorted(set(pos_labels))



In [ ]:
"""COLLECT DERIVATIONAL FORMS FROM GOLD JSON FILES"""

rows = []

for json_path in sorted(JSON_ROOT.rglob("*.json")):
    entry_file = json_path.name
    entry = json.loads(json_path.read_text(encoding="utf-8"))

    lemma = entry.get("lemma") or json_path.stem
    base_pos_values = pos_by_entry_file.get(entry_file, [])
    base_pos = ";".join(base_pos_values)


    flags = entry.get("flags") or {}
    is_obsolete = bool(flags.get("is_obsolete", False))
    is_archaic = bool(flags.get("is_archaic", False))
    is_dialectal = bool(flags.get("is_dialectal", False))

    for deriv in entry.get("derivations") or []:
        for form in deriv.get("form") or []:
            form = str(form).strip()
            if not form:
                continue

            rows.append({
                "entry_file": entry_file,
                "lemma": lemma,
                "base_pos": base_pos,
                "derivation_form": form,
                "is_obsolete": is_obsolete,
                "is_archaic": is_archaic,
                "is_dialectal": is_dialectal,
            })

deriv_df = pd.DataFrame(rows)

deriv_df["any_flag"] = (
    deriv_df["is_obsolete"]
    | deriv_df["is_archaic"]
    | deriv_df["is_dialectal"]
)

print(
    f"Collected {len(deriv_df):,} derivational forms "
    f"from {deriv_df['lemma'].nunique():,} lemmas."
)

display(deriv_df.sample(n=10, random_state=23).sort_index())


In [ ]:
"""RUN EMMORPH AND EXTRACT STEMS"""



all_forms = deriv_df["derivation_form"].dropna().unique().tolist()

print(f"Running emMorph on {len(all_forms):,} unique derivational forms...")

with contextlib.redirect_stdout(io.StringIO()):
    analyses = emmorph_analyse_many(
        all_forms,
        hfst_lookup=HFST_LOOKUP,
        hu_hfstol=HU_HFSTOL,
    )


deriv_df["extracted_stem"] = deriv_df.apply(
    lambda r: get_best_stem(r, analyses),
    axis=1)

deriv_df["stem_matches_lemma"] = (
    deriv_df["extracted_stem"].str.casefold()
    == deriv_df["lemma"].str.casefold())

print("Audit done.")


In [ ]:
"""SUMMARY STATISTICS"""

summary_rows = []

def summarize_emmorph_stem_recovery(subset, label):
    total = len(subset)
    covered = subset["extracted_stem"].notna().sum()
    matched = subset.loc[subset["extracted_stem"].notna(), "stem_matches_lemma"].sum()

    coverage_rate = covered / total if total else pd.NA
    accuracy_within_coverage = matched / covered if covered else pd.NA
    overall_accuracy = matched / total if total else pd.NA

    summary_rows.append({
        "subset": label,
        "total": total,
        "covered": covered,
        "coverage_rate": coverage_rate,
        "matched": matched,
        "accuracy_within_coverage": accuracy_within_coverage,
        "overall_accuracy": overall_accuracy,
    })

summarize_emmorph_stem_recovery(deriv_df, "All")
summarize_emmorph_stem_recovery(deriv_df[~deriv_df["any_flag"]], "No flag")
summarize_emmorph_stem_recovery(deriv_df[deriv_df["any_flag"]], "With flag")
summarize_emmorph_stem_recovery(deriv_df[deriv_df["is_obsolete"]], "Obsolete")
summarize_emmorph_stem_recovery(deriv_df[deriv_df["is_archaic"]], "Archaic")
summarize_emmorph_stem_recovery(deriv_df[deriv_df["is_dialectal"]], "Dialectal")

summary_df = pd.DataFrame(summary_rows)

display(summary_df.round(3))


## **SPLIT TABLES ABOUT THE MOST FREQUENT MATCHED, UNMATCHED, UNRECOGNIZED SUFFIXES**

In [ ]:
"""SPLIT MATCHED, UNMATCHED, AND NOT FOUND DERIVATIONS"""

found = deriv_df[deriv_df["extracted_stem"].notna() & deriv_df["stem_matches_lemma"]].copy()
failures = deriv_df[deriv_df["extracted_stem"].notna() & ~deriv_df["stem_matches_lemma"]].copy()
no_coverage = deriv_df[deriv_df["extracted_stem"].isna()].copy()

print(f"Matched:    {len(found):,}")
print(f"Unmatched:  {len(failures):,}")
print(f"Not found:  {len(no_coverage):,}")


In [ ]:
"""BUILD SUFFIX DATAFRAME FROM MATCHED FORMS"""



records = []

for _, row in found.iterrows():
    for surface, code in get_suffix_pairs_from_best_matching_analysis(row, analyses):
        records.append({
            "lemma": row["lemma"],
            "base_pos": row.get("base_pos", ""),
            "derivation_form": row["derivation_form"],
            "suffix_surface": surface,
            "emmorph_code": code,
        })


suffix_df = pd.DataFrame(records)
suffix_df["suffix_group"] = suffix_df["suffix_surface"].map(normalize_suffix)

print(
    f"suffix_df: {len(suffix_df):,} rows, "
    f"{suffix_df['suffix_group'].nunique()} canonical suffix variants" # such as -os/-es/-ös etc.
)


In [ ]:
"""ADD SIMPLE UNSTRIPPED SUFFIX"""

def get_unstripped_suffix(row):
    lemma = str(row["lemma"]).casefold()
    form = str(row["derivation_form"]).casefold()

    if form.startswith(lemma):
        return form[len(lemma):]

    return pd.NA

found["unstripped_suffix"] = found.apply(get_unstripped_suffix, axis=1)
failures["unstripped_suffix"] = failures.apply(get_unstripped_suffix, axis=1)
no_coverage["unstripped_suffix"] = no_coverage.apply(get_unstripped_suffix, axis=1)


In [ ]:
"""SAVE DETAILED EMMORPH AUDIT TABLES"""

deriv_df.to_csv(OUT_DIR / "derivation_stem_audit.csv", index=False, encoding="utf-8-sig",)
found.to_csv(OUT_DIR / "emmorph_found_examples.csv", index=False, encoding="utf-8-sig",)
failures.to_csv(OUT_DIR / "emmorph_no_matching_analysis_examples.csv", index=False, encoding="utf-8-sig",)
no_coverage.to_csv(OUT_DIR / "emmorph_no_coverage_suffix_examples.csv", index=False, encoding="utf-8-sig",)

print("emMorph audit CSVs found in", OUT_DIR)


In [ ]:
"""SUFFIX GROUP FREQUENCY FOR ALL DERIVATIONS

The saved CSV file has the frequency of all the matched,
not-found and unmatched derivations.
"""

all_suffix_rows = pd.concat([found.assign(status="matched"), failures.assign(status="unmatched"), no_coverage.assign(status="not_found"),], ignore_index=True)
all_suffix_rows["suffix_group"] = all_suffix_rows["unstripped_suffix"].map(normalize_suffix)

suffix_group_frequency = (
    all_suffix_rows
    .dropna(subset=["suffix_group"])
    .groupby(["status", "suffix_group"], dropna=False)
    .agg(count=("suffix_group", "size"), 
         examples=("derivation_form", lambda x: "; ".join(dict.fromkeys(x.astype(str).head(10)))))
    .reset_index().sort_values(["status", "count"], ascending=[True, False]))


suffix_group_frequency.to_csv(OUT_DIR / "emmorph_suffix_group_frequency.csv", index=False, encoding="utf-8-sig")
print("Wrote:", OUT_DIR / "emmorph_suffix_group_frequency.csv")


## **SUFFIX-LEVEL SUCCESS/FAILURE STATISTICS**

In [ ]:
"""ADD SIMPLE SUFFIX STRING FOR FAILED CASES"""

def get_unstripped_suffix(row):
    lemma = str(row["lemma"]).casefold()
    form = str(row["derivation_form"]).casefold()

    if form.startswith(lemma):
        return form[len(lemma):]

    return pd.NA


found["unstripped_suffix"] = found.apply(get_unstripped_suffix, axis=1)
failures["unstripped_suffix"] = failures.apply(get_unstripped_suffix, axis=1)
no_coverage["unstripped_suffix"] = no_coverage.apply(get_unstripped_suffix, axis=1)


In [ ]:
"""SUFFIX-LEVEL SUCCESS AND FAILURE STATISTICS"""

combined_failed_suffixes = pd.concat([failures["unstripped_suffix"], 
                                      no_coverage["unstripped_suffix"]], 
                                     ignore_index=True).map(normalize_suffix)

suffix_df["suffix_group"] = suffix_df["suffix_surface"].map(normalize_suffix)
clean_sf = clean_suffix_df(suffix_df)


def clean_suffix_series(s):
    return s[
        s.notna() # keep values that are not missing
        & (s.astype(str) != "None") # and keep values that are not None
        & (s.astype(str).str.strip() != "") # and keep values that are not empty
    ]


found_freq = (clean_suffix_series(clean_sf["suffix_group"]).value_counts().rename("found_count"))
not_found_freq = (clean_suffix_series(combined_failed_suffixes).value_counts().rename("not_found_count"))
code_lookup = (clean_sf.groupby("suffix_group")["emmorph_code"].apply(lambda x: "; ".join(sorted(set(x)))).to_dict())
suffix_stats = (found_freq.to_frame().join(not_found_freq, how="outer").fillna(0).astype(int))

suffix_stats["total"] = suffix_stats["found_count"] + suffix_stats["not_found_count"]
suffix_stats["success_rate"] = suffix_stats["found_count"] / suffix_stats["total"]
suffix_stats["failure_rate"] = suffix_stats["not_found_count"] / suffix_stats["total"]

suffix_stats = (suffix_stats.query("total >= 20").reset_index().rename(columns={"index": "suffix"}))
suffix_stats["emmorph_codes"] = suffix_stats["suffix"].map(code_lookup)
suffix_stats["gloss"] = suffix_stats["suffix"].map(SUFFIX_GLOSSES).fillna("")
suffix_stats = suffix_stats.sort_values("total", ascending=False)

worst_suffixes = (suffix_stats.sort_values(["failure_rate", "total"], ascending=[False, False]).head(10))
best_suffixes = (suffix_stats.sort_values(["success_rate", "total"], ascending=[False, False]).head(10))

print("Worst suffixes (highest failure rate):")
display(worst_suffixes[[
    "suffix",
    "found_count",
    "not_found_count",
    "total",
    "failure_rate",
    "emmorph_codes",
    "gloss",
]])

print("Best suffixes (highest success rate):")
display(best_suffixes[[
    "suffix",
    "found_count",
    "not_found_count",
    "total",
    "success_rate",
    "emmorph_codes",
    "gloss",
]])

print("All suffix groups among the grouped ones:")
display(suffix_stats[[
    "suffix",
    "found_count",
    "not_found_count",
    "total",
    "success_rate",
    "failure_rate",
    "emmorph_codes",
    "gloss",
]])


## **PREVERB ANALYSIS**

In [ ]:
"""PREVERB ANALYSIS DIAGNOSTIC"""


prev_info = deriv_df.apply(
    lambda r: classify_preverb_analyses(r, analyses),
    axis=1,
    result_type="expand",
).rename(columns={
    "category": "preverb_category",
    "preverb": "preverb_candidate",
    "stem_after_prev": "stem_after_preverb",
})

deriv_prev = pd.concat([deriv_df.copy(), prev_info], axis=1)

deriv_prev["match_status"] = np.select(
    [
        deriv_prev["stem_matches_lemma"].fillna(False),
        deriv_prev["extracted_stem"].isna(),
    ],
    [
        "matched",
        "no_coverage",
    ],
    default="unmatched",
)

preverb_order = ["none", "valid", "false_spurious", "false_atomic"]

preverb_counts = (
    deriv_prev["preverb_category"]
    .value_counts()
    .reindex(preverb_order, fill_value=0)
)

preverb_summary = pd.DataFrame({
    "count": preverb_counts,
    "percent": 100 * preverb_counts / len(deriv_prev),
})

preverb_by_status = (
    pd.crosstab(
        deriv_prev["preverb_category"],
        deriv_prev["match_status"],
    )
    .reindex(preverb_order)
    .fillna(0)
    .astype(int)
)

display(preverb_summary.round(2))
display(preverb_by_status)


In [ ]:
"""PREVERB ANALYSIS EXAMPLES"""

cols = [
    "lemma",
    "derivation_form",
    "preverb_candidate",
    "stem_after_preverb",
    "extracted_stem",
    "stem_matches_lemma",
    "match_status",
]

""" PSEUDE-SUFFIXES, NOT REAL ONES """
false_preverbs = (
    deriv_prev
    .loc[deriv_prev["preverb_category"] == "false_spurious", cols]
    .sort_values(["preverb_candidate", "lemma", "derivation_form"])
)

""" REAL SUFFIXES, BUT EMMORPH DOES NOT USE THE SAME ANALYSIS AS UESZ """
atomic_preverbs = (
    deriv_prev
    .loc[deriv_prev["preverb_category"] == "false_atomic", cols]
    .sort_values(["preverb_candidate", "lemma", "derivation_form"])
)

""" MATCHING ANALYSIS WITH UESZ """
valid_preverbs = (
    deriv_prev
    .loc[deriv_prev["preverb_category"] == "valid", cols]
    .sort_values(["preverb_candidate", "lemma", "derivation_form"])
)

print("False preverb candidates (FP):")
display(false_preverbs)

print("Genuine preverb, but post-preverb stem does not match lemma (FN):")
display(atomic_preverbs)

print("Genuine preverb and lemma recovered (TP):")
display(valid_preverbs)
